# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** In Croissant, *record sets* refer to tabular data entities, each described by a unique `@id`. Within each record set there are *fields*, each also identified by their own `@id`. We'll enumerate all available record sets, and for each, print their fields and types.

In [ ]:
# List the available record sets in the dataset (by @id and name)
record_sets = list(dataset.record_sets)
if record_sets:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']}  name={rs.get('name', '<no name>')}")
        # List fields for each record set
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                if isinstance(field, dict):
                    print(f"   - @id={field.get('@id', '')} name={field.get('name', '')} (type={field.get('dataType', '')})")
                else:
                    print(f"   - @id={field}")
        print()
else:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Here, we load all tables present (by their `@id`). **All subsequent references to record sets and fields are via their `@id`s**, following best practices for reproducibility.

In [ ]:
# Gather a list of record set @ids
record_set_ids = [rs['@id'] for rs in getattr(dataset, 'record_sets', [])]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records. Columns: {list(df.columns.values)}")
        else:
            print(f"  [!] No records found for @id={record_set_id}.")
    except Exception as e:
        print(f"  [!] Could not load records for {record_set_id}: {e}")

# Example inspection: Show columns in the first DataFrame
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns for record set '@id={first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No data loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's perform initial processing on the loaded records (if present): filtering, normalization, and grouping.

- We'll select a **numeric field** from the first loaded record set for filtering and normalization.
- We'll choose a **categorical/group field** for aggregation if available.

Again, all fields/columns are referenced by their `@id`.

In [ ]:
import numpy as np
from pandas.api.types import is_numeric_dtype

if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    
    # Select a numeric field (by @id or column name)
    numeric_field = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to convert columns to numeric and pick the first with >10 unique numeric values
        for col in df.columns:
            try:
                series = pd.to_numeric(df[col], errors='coerce')
                if series.notnull().sum() > 10:
                    df[col] = series
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        threshold = np.nanmedian(df[numeric_field]) if np.issubdtype(df[numeric_field].dtype, np.number) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick a group field (candidate: first non-numeric)
        group_field = None
        for col in df.columns:
            if not is_numeric_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        
        # Group and aggregate by group field
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
            print(f"\nGrouped mean {numeric_field} by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("DataFrames not available; cannot perform EDA.")

## 5. Visualization
Visualize the results:

- If a numeric field was found in EDA, we'll plot its distribution (histogram), as well as the means per group if a group field was found.
- The field names correspond to their `@id`s or source columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=25)
    plt.title(f"Distribution of {numeric_field} in record set '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        grouped_df['mean_'+numeric_field].plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric data available to visualize.")

## 6. Conclusion

This notebook has demonstrated how to:

- Load and inspect a Croissant-based dataset using the `mlcroissant` library with all references by their `@id`
- Explore available record sets, fields, and column structure
- Extract and analyze tabular data using DataFrames
- Filter and normalize numeric fields, group by categorical variables, and create simple visualizations

Further analyses can now be performed on the raw or processed data, for statistical modeling or further data science workflows.